In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from selenium import webdriver
%load_ext autoreload
%autoreload 2
import json
import time
import duckdb
import importlib
import numpy as np
import numbers
import numpy as np
import pandas as pd
from pathlib import Path
import json
from mplsoccer import Pitch, VerticalPitch
import matplotlib.pyplot as plt
from plottable import ColumnDefinition, Table
from plottable.cmap import normed_cmap
from matplotlib.colors import PowerNorm
from matplotlib.lines import Line2D
from soccerplots.radar_chart import Radar
from mplsoccer import grid
from matplotlib.lines import Line2D
from mplsoccer import VerticalPitch
import matplotlib.patches as mpatches
import math

from great_tables import GT

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Create metrics class
        
        Subtask: simple xG model

Create Z-Score Creation Class

Create Export Table Class

Forwards
    npXg
    opXa
    *shots*
    *npGoals*
    Aerial %
    *EPV passing*
    *Dribble/Takeon rate*
    Carrying
    Touches in penalty area
    Through balls received
    *Through balls player*
    Defensive actions in final third
    Fouls won

Midfielders
    *Progressive pass distance*
    *Progressive passes*
    Progressive carries
    *Passes into penalty area*
    Total tackles
    Successful tackles
    Tackle %
    *Pass completion %*
    *Forward passes*
    *Forward pass %*
    Interceptions (tackles and interceptions?)
    *Touches*


Defenders
    Blocked shots
    Box clearances
    Recoveries

In [ ]:
class Metrics:
    def __init__(self, file_path: str, conditions:str = None):
        """
        file_path is path to DuckDB database. Conditions is a string representing the "where" clause if specific ranges are needed
        """
        con = duckdb.connect('file_path')
        if conditions == None:
            self.events = con.sql('select * from events').df()
        else:
            self.events = con.sql(f'select * from players where {conditions}')
        self.players = con.sql('select * from players').df()

    def create_metrics(self):
        
        df_nonpenalty = self.events[self.events['penaltyScore'] == False]
        
        shooting_agg = (
            df_nonpenalty
            .groupby(['playerId'])
            .agg(
                shots = ('isShot', 'sum'),
                np_goals = ('isGoal', 'sum')
            )
        )

        dribbling_agg = (
            self.events
            .groupby(['playerId'])
            .agg(
                dribbles_lost = ('dribbleLost', 'sum'),
                dribbles_won = ('dribbleWon', 'sum'),
                touches= ('isTouch' , 'sum')
            )
        )

        dribbling_agg['dribbles_total'] = (dribbling_agg['dribbles_lost'] + dribbling_agg['dribbles_won']) #total dribbles always shows zero in some games so I have to do it this way
        dribbling_agg['dribble_success'] = (dribbling_agg['dribbles_won'] / (dribbling_agg['dribbles_total']))*100

        passing_df = self.events[self.events['type'] == 'Pass']
        passing_df['isKeyPassOP'] = passing_df['keyPassLong', 'keyPassShort', 'keyPassCross', 'keyPassThroughball', 'keyPassOther'].any(axis=1)
        
        passing_agg_basic = (
            passing_df
            .groupby(['playerId'])
            .agg(
                key_passes_open_play = ('isKeyPassOP', 'sum'),
                through_balls_completed = ('passThroughBallAccurate', 'sum'),
                passes = ('isTouch', 'sum'), #this is weird but it works
                completed_passes = ('outcomeType', lambda x: x == 'Successful'),
                pass_epv = ('EPV', 'sum'),
                forward_passes = ('passForward', 'sum')
            )
        )
        
        passing_agg_basic['completion_percentage'] = (passing_agg_basic['completed_passes'] / passing_agg_basic['passes']) * 100
        passing_agg_basic['forward_pct'] = (passing_agg_basic['passForward'] / passing_agg_basic['passes']) * 100
        
        passing_advanced_df = self.events[self.events['type'] == 'Pass']
        passing_advanced_df = passing_advanced_df[passing_advanced_df['outcomeType'] == 'True']

        passing_advanced_df['progressiveDistance'] =  passing_advanced_df['endX'] - passing_advanced_df['x']
        passing_advanced_df['progressiveDistance'] =  passing_advanced_df['progressive_distance'].clip(lower = 0.0)

        passing_advanced_df['dist_start'] = np.sqrt((100 - passing_advanced_df['x'])**2 + (50 - passing_advanced_df['y'])**2)
        passing_advanced_df['dist_end'] = np.sqrt((100 - passing_advanced_df['endX'])**2 + (50 - passing_advanced_df['endY'])**2)

        passing_advanced_df['isProgressivePass'] = passing_advanced_df['x'] > 33.333333 and passing_advanced_df['dist_start'] <= 0.75 * passing_advanced_df['dist_end']
       
        passing_advanced_df['penAreaEnd'] = passing_advanced_df['x'] >= 83 and passing_advanced_df['y'] >= 21 and passing_advanced_df['y'] <= 79

        passing_agg_advanced = (
            passing_advanced_df
            .groupby(['playerId'])
            .agg(
                progressive_distance = ('progressiveDistance', 'sum'),
                progressive_passes = ('isProgressivePass', 'sum'),
                passes_into_pen_area = ('penAreaEnd', 'sum')
            )
        )

        
        
        
        
        


        

    
        